# AI in Cybersecurity (ICT4416) - Internal Assessment 4
# Network Intrusion Detection System (NIDS)

Binary classification of network traffic as Normal or Attack using the UNSW-NB15 dataset.
15 features (3 categorical, 12 numerical) and 1 binary target label.

## 1. Exploratory Data Analysis

This section examines the UNSW-NB15 dataset structure, distributions, correlations, and feature characteristics prior to model building.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             accuracy_score, f1_score)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import warnings
warnings.filterwarnings('ignore')

# Device-agnostic setup: works on CPU, GPU (CUDA), and Apple Silicon (MPS)
print(f"TensorFlow version: {tf.__version__}")
physical_devices = tf.config.list_physical_devices()
print(f"Available devices: {[d.device_type for d in physical_devices]}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU(s) detected: {len(gpus)}. Memory growth enabled.")
else:
    print("No GPU detected. Using CPU.")

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
PALETTE = sns.color_palette('Set2')
LABEL_COLORS = {0: PALETTE[0], 1: PALETTE[1]}
LABEL_NAMES = {0: 'Normal', 1: 'Attack'}

print(f"Pandas version: {pd.__version__}")

In [ ]:
# Load datasets
train_df = pd.read_csv('UNSW_NB15_train_40k.csv')
test_df = pd.read_csv('UNSW_NB15_test_10k.csv')

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape:     {test_df.shape}")
print(f"\nColumn names:\n{list(train_df.columns)}")

In [ ]:
# Data types
print("=== Training Set Data Types ===")
print(train_df.dtypes)
print(f"\nCategorical columns: {list(train_df.select_dtypes(include='object').columns)}")
print(f"Numerical columns:   {list(train_df.select_dtypes(include=np.number).columns)}")

In [ ]:
# Info for both datasets
print("=== Training Set Info ===")
train_df.info()
print("\n=== Test Set Info ===")
test_df.info()

In [ ]:
# Descriptive statistics for training set
train_df.describe(include='all').T

In [ ]:
# First rows
print("=== Training Set: First 5 Rows ===")
display(train_df.head())
print("\n=== Test Set: First 5 Rows ===")
display(test_df.head())

### 1.1 Missing Value Analysis

In [ ]:
# Missing value counts
train_missing = train_df.isnull().sum()
test_missing = test_df.isnull().sum()

print("=== Training Set Missing Values ===")
print(train_missing[train_missing > 0] if train_missing.sum() > 0 else "No missing values found.")
print(f"\nTotal missing cells (train): {train_missing.sum()}")

print("\n=== Test Set Missing Values ===")
print(test_missing[test_missing > 0] if test_missing.sum() > 0 else "No missing values found.")
print(f"\nTotal missing cells (test): {test_missing.sum()}")

In [ ]:
# Missing value heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(train_df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Missing Values Heatmap (Training Set)', fontsize=13)
axes[0].set_xlabel('Features')

sns.heatmap(test_df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=axes[1])
axes[1].set_title('Missing Values Heatmap (Test Set)', fontsize=13)
axes[1].set_xlabel('Features')

plt.tight_layout()
plt.show()

### 1.2 Class Distribution

In [ ]:
# Class distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (df, name) in enumerate([(train_df, 'Training'), (test_df, 'Test')]):
    counts = df['label'].value_counts().sort_index()
    bars = axes[idx].bar(
        [LABEL_NAMES[i] for i in counts.index],
        counts.values,
        color=[LABEL_COLORS[i] for i in counts.index],
        edgecolor='black', linewidth=0.8
    )
    for bar, val in zip(bars, counts.values):
        axes[idx].annotate(f'{val:,}\n({val/len(df)*100:.1f}%)',
                           xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                           ha='center', va='bottom', fontsize=12, fontweight='bold')
    axes[idx].set_title(f'Class Distribution ({name} Set)', fontsize=13)
    axes[idx].set_ylabel('Count')
    axes[idx].set_xlabel('Label')

plt.tight_layout()
plt.show()

print(f"Training set class ratio (Attack/Normal): {train_df['label'].value_counts()[1] / train_df['label'].value_counts()[0]:.3f}")
print(f"Test set class ratio (Attack/Normal):     {test_df['label'].value_counts()[1] / test_df['label'].value_counts()[0]:.3f}")

### 1.3 Categorical Feature Analysis

In [ ]:
# Value counts for categorical features
categorical_cols = ['proto', 'state', 'service']

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(f"Unique values (train): {train_df[col].nunique()}")
    print(f"Unique values (test):  {test_df[col].nunique()}")
    print(f"\nTop 10 values (train):")
    print(train_df[col].value_counts().head(10))

In [ ]:
# Bar charts for categorical feature value counts (top 15)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, col in enumerate(categorical_cols):
    top_vals = train_df[col].value_counts().head(15)
    bars = axes[idx].barh(
        top_vals.index[::-1], top_vals.values[::-1],
        color=PALETTE[idx], edgecolor='black', linewidth=0.5
    )
    for bar, val in zip(bars, top_vals.values[::-1]):
        axes[idx].text(bar.get_width() + max(top_vals.values)*0.01, bar.get_y() + bar.get_height()/2,
                       f'{val:,}', va='center', fontsize=9)
    axes[idx].set_title(f'Top {min(15, len(top_vals))} Values: {col}', fontsize=13)
    axes[idx].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar charts: categorical feature distribution by label
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for idx, col in enumerate(categorical_cols):
    top_categories = train_df[col].value_counts().head(10).index
    subset = train_df[train_df[col].isin(top_categories)]

    ct = pd.crosstab(subset[col], subset['label'])
    ct.columns = ['Normal', 'Attack']
    ct = ct.loc[top_categories]

    ct.plot(kind='barh', ax=axes[idx], color=[LABEL_COLORS[0], LABEL_COLORS[1]],
            edgecolor='black', linewidth=0.5)
    axes[idx].set_title(f'{col}: Distribution by Label (Top 10)', fontsize=13)
    axes[idx].set_xlabel('Count')
    axes[idx].set_ylabel(col)
    axes[idx].legend(title='Label')

plt.tight_layout()
plt.show()

In [ ]:
# Stacked percentage bar charts: attack rate per category
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for idx, col in enumerate(categorical_cols):
    top_categories = train_df[col].value_counts().head(10).index
    subset = train_df[train_df[col].isin(top_categories)]

    ct = pd.crosstab(subset[col], subset['label'], normalize='index') * 100
    ct.columns = ['Normal %', 'Attack %']
    ct = ct.loc[top_categories]

    ct.plot(kind='barh', stacked=True, ax=axes[idx],
            color=[LABEL_COLORS[0], LABEL_COLORS[1]],
            edgecolor='black', linewidth=0.5)
    axes[idx].set_title(f'{col}: Attack Rate by Category (Top 10)', fontsize=13)
    axes[idx].set_xlabel('Percentage (%)')
    axes[idx].set_ylabel(col)
    axes[idx].legend(title='Label')
    axes[idx].set_xlim(0, 100)

plt.tight_layout()
plt.show()

### 1.4 Numerical Feature Distributions

In [ ]:
numerical_cols = ['dur', 'sbytes', 'dbytes', 'spkts', 'dpkts', 'sload', 'dload',
                  'sttl', 'dttl', 'smean', 'dmean', 'sinpkt']

print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")

In [ ]:
# Overlaid histograms for all numerical features, colored by label
fig, axes = plt.subplots(4, 3, figsize=(20, 22))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    for label_val in [0, 1]:
        data = train_df[train_df['label'] == label_val][col]
        axes[idx].hist(data, bins=50, alpha=0.6, label=LABEL_NAMES[label_val],
                       color=LABEL_COLORS[label_val], edgecolor='black', linewidth=0.3)
    axes[idx].set_title(f'Distribution of {col}', fontsize=12)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend(fontsize=9)
    axes[idx].ticklabel_format(style='scientific', axis='y', scilimits=(0, 3))

plt.suptitle('Numerical Feature Distributions by Label (Training Set)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# KDE plots for numerical features by label
fig, axes = plt.subplots(4, 3, figsize=(20, 22))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    for label_val in [0, 1]:
        data = train_df[train_df['label'] == label_val][col]
        if data.std() > 0:
            sns.kdeplot(data, ax=axes[idx], label=LABEL_NAMES[label_val],
                        color=LABEL_COLORS[label_val], fill=True, alpha=0.3)
    axes[idx].set_title(f'KDE: {col}', fontsize=12)
    axes[idx].set_xlabel(col)
    axes[idx].legend(fontsize=9)

plt.suptitle('Kernel Density Estimates by Label (Training Set)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for all numerical features grouped by label
fig, axes = plt.subplots(4, 3, figsize=(20, 22))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    sns.boxplot(x='label', y=col, data=train_df, ax=axes[idx],
                palette=LABEL_COLORS, showfliers=False)
    axes[idx].set_title(f'Box Plot: {col} by Label', fontsize=12)
    axes[idx].set_xticklabels(['Normal', 'Attack'])
    axes[idx].set_xlabel('Label')
    axes[idx].set_ylabel(col)

plt.suptitle('Numerical Feature Box Plots by Label (Outliers Hidden for Clarity)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots for all numerical features
fig, axes = plt.subplots(4, 3, figsize=(20, 22))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    sns.violinplot(x='label', y=col, data=train_df, ax=axes[idx],
                   palette=LABEL_COLORS, inner='quartile', cut=0)
    axes[idx].set_title(f'Violin Plot: {col} by Label', fontsize=12)
    axes[idx].set_xticklabels(['Normal', 'Attack'])
    axes[idx].set_xlabel('Label')
    axes[idx].set_ylabel(col)

plt.suptitle('Numerical Feature Violin Plots by Label (Training Set)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 1.5 Skewness and Kurtosis Analysis

In [ ]:
# Compute skewness for all numerical features
skew_values = train_df[numerical_cols].skew().sort_values(ascending=True)
print("=== Skewness Values ===")
for col, val in skew_values.items():
    flag = " [highly skewed]" if abs(val) > 2 else ""
    print(f"  {col:>10s}: {val:>10.3f}{flag}")

In [ ]:
# Horizontal bar chart of skewness
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['#e74c3c' if abs(v) > 2 else '#3498db' if abs(v) > 1 else '#2ecc71' for v in skew_values.values]
bars = ax.barh(skew_values.index, skew_values.values, color=colors, edgecolor='black', linewidth=0.5)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.axvline(x=2, color='red', linewidth=0.8, linestyle='--', alpha=0.5, label='|skew| = 2')
ax.axvline(x=-2, color='red', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(x=1, color='orange', linewidth=0.8, linestyle='--', alpha=0.5, label='|skew| = 1')
ax.axvline(x=-1, color='orange', linewidth=0.8, linestyle='--', alpha=0.5)

for bar, val in zip(bars, skew_values.values):
    ax.text(bar.get_width() + 0.1 if val >= 0 else bar.get_width() - 0.1,
            bar.get_y() + bar.get_height()/2, f'{val:.2f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=10)

ax.set_title('Skewness of Numerical Features', fontsize=14)
ax.set_xlabel('Skewness')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Kurtosis analysis
kurt_values = train_df[numerical_cols].kurtosis().sort_values(ascending=True)
print("=== Kurtosis Values ===")
for col, val in kurt_values.items():
    flag = " [heavy tails]" if val > 7 else ""
    print(f"  {col:>10s}: {val:>12.3f}{flag}")

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#e74c3c' if v > 7 else '#3498db' for v in kurt_values.values]
ax.barh(kurt_values.index, kurt_values.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Kurtosis of Numerical Features', fontsize=14)
ax.set_xlabel('Kurtosis')
plt.tight_layout()
plt.show()

### 1.6 Correlation Analysis

In [ ]:
# Correlation matrix (lower triangle)
corr_cols = numerical_cols + ['label']
corr_matrix = train_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap (Numerical Features + Label)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Full correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})
ax.set_title('Full Correlation Matrix (Numerical Features + Label)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with label
label_corr = corr_matrix['label'].drop('label').sort_values(ascending=True)

print("=== Correlations with Label ===")
for col, val in label_corr.items():
    print(f"  {col:>10s}: {val:>7.4f}")

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in label_corr.values]
bars = ax.barh(label_corr.index, label_corr.values, color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, label_corr.values):
    ax.text(bar.get_width() + 0.005 if val >= 0 else bar.get_width() - 0.005,
            bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=10)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Label (Target)', fontsize=14)
ax.set_xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 feature-feature correlations
upper_tri = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))
corr_pairs = upper_tri.stack().reset_index()
corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
corr_pairs['abs_corr'] = corr_pairs['Correlation'].abs()
top_pairs = corr_pairs.sort_values('abs_corr', ascending=False).head(15)

print("=== Top 15 Feature Correlations (by absolute value) ===")
for _, row in top_pairs.iterrows():
    print(f"  {row['Feature 1']:>10s} <-> {row['Feature 2']:<10s}: {row['Correlation']:>7.4f}")

### 1.7 Pairwise Scatter Plot Matrix

In [ ]:
# Select top 5 features most correlated with label
top_5_features = label_corr.abs().sort_values(ascending=False).head(5).index.tolist()
print(f"Top 5 features correlated with label: {top_5_features}")

# Pair plot colored by label (sample for speed)
plot_df = train_df[top_5_features + ['label']].sample(n=min(5000, len(train_df)), random_state=42)
plot_df['label'] = plot_df['label'].map(LABEL_NAMES)

g = sns.pairplot(plot_df, hue='label', palette={'Normal': PALETTE[0], 'Attack': PALETTE[1]},
                 diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15, 'edgecolor': 'none'},
                 height=2.5)
g.figure.suptitle('Pairwise Scatter Matrix: Top 5 Label-Correlated Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots for the top 3 most correlated feature pairs
top_3_pairs = corr_pairs.sort_values('abs_corr', ascending=False).head(3)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (_, row) in enumerate(top_3_pairs.iterrows()):
    f1, f2, corr_val = row['Feature 1'], row['Feature 2'], row['Correlation']
    sample = train_df.sample(n=min(5000, len(train_df)), random_state=42)

    for label_val in [0, 1]:
        m = sample['label'] == label_val
        axes[idx].scatter(sample.loc[m, f1], sample.loc[m, f2],
                         alpha=0.3, s=10, label=LABEL_NAMES[label_val],
                         color=LABEL_COLORS[label_val], edgecolors='none')
    axes[idx].set_xlabel(f1, fontsize=11)
    axes[idx].set_ylabel(f2, fontsize=11)
    axes[idx].set_title(f'{f1} vs {f2} (r={corr_val:.3f})', fontsize=12)
    axes[idx].legend(fontsize=9)

plt.suptitle('Top 3 Correlated Feature Pairs, Colored by Label', fontsize=14)
plt.tight_layout()
plt.show()

### 1.8 Summary Statistics by Class

In [ ]:
# Grouped statistics by label
print("=== Mean Values by Label ===")
grouped_means = train_df.groupby('label')[numerical_cols].mean().T
grouped_means.columns = ['Normal (mean)', 'Attack (mean)']
grouped_means['Difference'] = grouped_means['Attack (mean)'] - grouped_means['Normal (mean)']
grouped_means['Ratio'] = grouped_means['Attack (mean)'] / grouped_means['Normal (mean)'].replace(0, np.nan)
print(grouped_means.round(3).to_string())

In [ ]:
# Heatmap of normalized mean feature values by class
norm_means = train_df.groupby('label')[numerical_cols].mean()
scaler_viz = MinMaxScaler()
norm_means_scaled = pd.DataFrame(
    scaler_viz.fit_transform(norm_means.T).T,
    index=['Normal', 'Attack'],
    columns=numerical_cols
)

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(norm_means_scaled, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Normalized Mean Feature Values by Class', fontsize=14)
ax.set_ylabel('Class')
plt.tight_layout()
plt.show()

### 1.9 EDA Observations

**Dataset structure:** 40,000 training samples and 10,000 test samples, each with 15 features (3 categorical, 12 numerical) and 1 binary target.

**Class balance:** The label distribution should be evaluated for imbalance. Any ratio above 1.5:1 will require handling during model training through class weights or resampling.

**Categorical features:** proto, state, and service have varying cardinality. Some categories appear almost exclusively in one class, making them useful discriminators.

**Numerical distributions:** Several features (sbytes, dbytes, sload, dload, smean, dmean) show heavy right skew with many near-zero values and long tails. Attack and normal traffic have distinct distribution shapes for these features.

**Skewness:** Features with |skewness| > 2 may benefit from transformations before feeding into distance-based or gradient-based models.

**Correlations:** The heatmap reveals which features carry redundant information (high inter-feature correlation) and which are most predictive of the label. Features with near-zero label correlation may contribute noise rather than signal.

## 2. Data Preprocessing

This section transforms raw features into a form suitable for ML classifiers. Each step includes justification and before/after comparisons.

In [ ]:
# Make working copies to preserve originals
train = train_df.copy()
test = test_df.copy()

### 2.1 Missing Value Assessment

Missing values can cause errors in most sklearn estimators and bias learned parameters. Both sets are checked; median imputation is used for numerical columns (robust to skew) and mode imputation for categorical columns.

In [ ]:
# Check missing values in both sets
train_missing = train.isnull().sum()
test_missing = test.isnull().sum()

missing_report = pd.DataFrame({
    'Train Missing': train_missing[train_missing > 0],
    'Test Missing': test_missing[test_missing > 0]
}).fillna(0).astype(int)

if missing_report.empty:
    print("No missing values found in either set.")
else:
    print("Missing values before imputation:")
    display(missing_report)

In [ ]:
categorical_cols = ['proto', 'state', 'service']
numerical_cols = ['dur', 'sbytes', 'dbytes', 'spkts', 'dpkts',
                  'sload', 'dload', 'sttl', 'dttl', 'smean', 'dmean', 'sinpkt']

# Impute numerical columns with median (robust to outliers/skew)
for col in numerical_cols:
    if train[col].isnull().any() or test[col].isnull().any():
        median_val = train[col].median()
        train[col].fillna(median_val, inplace=True)
        test[col].fillna(median_val, inplace=True)

# Impute categorical columns with mode
for col in categorical_cols:
    if train[col].isnull().any() or test[col].isnull().any():
        mode_val = train[col].mode()[0]
        train[col].fillna(mode_val, inplace=True)
        test[col].fillna(mode_val, inplace=True)

# Verify
total_after = train.isnull().sum().sum() + test.isnull().sum().sum()
print(f"Total missing values after imputation: {total_after}")

### 2.2 Categorical Encoding

The three categorical features (proto, state, service) have varying cardinality. **Label Encoding** is used rather than One-Hot Encoding for two reasons:

1. `service` has high cardinality; one-hot encoding would create dozens of sparse columns, increasing dimensionality and memory usage.
2. Tree-based models handle ordinal integers natively. For distance-based models, the subsequent scaling step mitigates arbitrary ordinal distances.

Categories present in the test set but absent from training are mapped to a dedicated 'unknown' integer to prevent encoding errors.

In [ ]:
# Inspect unique values in both sets
for col in categorical_cols:
    train_unique = set(train[col].unique())
    test_unique = set(test[col].unique())
    unseen = test_unique - train_unique
    print(f"[{col}]  Train unique: {len(train_unique)}  |  Test unique: {len(test_unique)}  |  Unseen in test: {len(unseen)}")
    if unseen:
        print(f"         Unseen categories: {unseen}")
    print()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    known_classes = list(train[col].unique())
    le.fit(known_classes + ['__unseen__'])

    # Map unseen test categories to the placeholder before transforming
    test[col] = test[col].apply(lambda x: x if x in known_classes else '__unseen__')

    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])
    label_encoders[col] = le

print("Encoded dtypes:")
print(train[categorical_cols].dtypes)
print()
print("Sample encoded values (train):")
display(train[categorical_cols].head())

### 2.3 Outlier Detection and Capping (IQR Method)

Network traffic features are heavily right-skewed; a small number of flows carry very large byte counts or durations. Removing these rows would discard legitimate attack signatures, so values are **clipped** to the IQR fences instead. This preserves all samples while reducing the influence of extreme values on distance-based learners and gradient updates.

Fence definition: lower = Q1 - 1.5 * IQR, upper = Q3 + 1.5 * IQR.

In [ ]:
# Box plots BEFORE capping
plot_features = ['dur', 'sbytes', 'dbytes', 'sload', 'dload', 'smean']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Numerical Features BEFORE Outlier Capping', fontsize=14)
for ax, feat in zip(axes.ravel(), plot_features):
    ax.boxplot(train[feat].dropna(), vert=True)
    ax.set_title(feat)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# IQR-based capping (fitted on train statistics only)
clip_bounds = {}

for col in numerical_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    clip_bounds[col] = (lower, upper)

    train[col] = train[col].clip(lower=lower, upper=upper)
    test[col] = test[col].clip(lower=lower, upper=upper)

print("Clip bounds (lower, upper) per feature:")
for col, (lo, hi) in clip_bounds.items():
    print(f"  {col:>10s}: [{lo:.4f}, {hi:.4f}]")

In [ ]:
# Box plots AFTER capping
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Numerical Features AFTER Outlier Capping', fontsize=14)
for ax, feat in zip(axes.ravel(), plot_features):
    ax.boxplot(train[feat].dropna(), vert=True)
    ax.set_title(feat)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### 2.4 Feature Scaling (StandardScaler)

StandardScaler (zero mean, unit variance) is applied to all numerical features. The scaler is **fit on the training set only** and then applied to both sets. This prevents information leakage from the test distribution into the training pipeline.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Store unscaled copies for before/after comparison
train_unscaled = train[numerical_cols].copy()

scaler = StandardScaler()
train[numerical_cols] = scaler.fit_transform(train[numerical_cols])
test[numerical_cols] = scaler.transform(test[numerical_cols])

print("Post-scaling train statistics (should be ~0 mean, ~1 std):")
print(train[numerical_cols].describe().loc[['mean', 'std']].round(4))

In [ ]:
# Before/after distribution comparison for selected features
compare_feats = ['dur', 'sbytes', 'sload', 'smean']

fig, axes = plt.subplots(len(compare_feats), 2, figsize=(14, 3.5 * len(compare_feats)))

for i, feat in enumerate(compare_feats):
    axes[i, 0].hist(train_unscaled[feat], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i, 0].set_title(f'{feat} - Before Scaling')
    axes[i, 0].set_ylabel('Count')

    axes[i, 1].hist(train[feat], bins=50, color='darkorange', edgecolor='black', alpha=0.7)
    axes[i, 1].set_title(f'{feat} - After Scaling')
    axes[i, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### 2.5 Feature Selection via Mutual Information

Mutual information measures the dependency between each feature and the target without assuming a linear relationship. Features with near-zero MI contribute mostly noise. MI scores are computed, visualized, and used to decide whether to drop any features.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

feature_cols = [c for c in train.columns if c != 'label']

mi_scores = mutual_info_classif(
    train[feature_cols], train['label'], discrete_features='auto', random_state=42
)

mi_df = pd.DataFrame({
    'Feature': feature_cols,
    'MI Score': mi_scores
}).sort_values('MI Score', ascending=False).reset_index(drop=True)

print(mi_df.to_string(index=False))

In [ ]:
# Bar chart of mutual information scores
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(mi_df['Feature'], mi_df['MI Score'], color='teal', edgecolor='black')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Importance (Mutual Information with Target)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Drop features with negligible MI (threshold: 0.01)
mi_threshold = 0.01
low_mi = mi_df[mi_df['MI Score'] < mi_threshold]['Feature'].tolist()

if low_mi:
    print(f"Dropping {len(low_mi)} feature(s) with MI < {mi_threshold}: {low_mi}")
    train.drop(columns=low_mi, inplace=True)
    test.drop(columns=low_mi, inplace=True)
else:
    print(f"All features have MI >= {mi_threshold}. Keeping all {len(feature_cols)} features.")

selected_features = [c for c in train.columns if c != 'label']
print(f"\nFeatures retained: {len(selected_features)}")

### 2.6 Class Imbalance Handling (SMOTE)

Class imbalance biases classifiers toward the majority class, producing high accuracy but poor recall on the minority class. SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic minority samples by interpolating between existing minority neighbors. It is applied **only to the training set**; the test set remains untouched to reflect real-world class proportions.

In [ ]:
from imblearn.over_sampling import SMOTE

X_train_pre = train.drop(columns=['label'])
y_train_pre = train['label']

# Class distribution before SMOTE
before_counts = y_train_pre.value_counts().sort_index()
print("Class distribution BEFORE SMOTE:")
print(before_counts)
print()

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_pre, y_train_pre)

after_counts = pd.Series(y_train_resampled).value_counts().sort_index()
print("Class distribution AFTER SMOTE:")
print(after_counts)

In [ ]:
# Side-by-side bar chart: before vs after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

axes[0].bar(['Normal (0)', 'Attack (1)'], before_counts.values,
            color=['#4c72b0', '#dd8452'], edgecolor='black')
axes[0].set_title('Before SMOTE')
axes[0].set_ylabel('Sample Count')
for j, v in enumerate(before_counts.values):
    axes[0].text(j, v + 100, str(v), ha='center', fontweight='bold')

axes[1].bar(['Normal (0)', 'Attack (1)'], after_counts.values,
            color=['#4c72b0', '#dd8452'], edgecolor='black')
axes[1].set_title('After SMOTE')
for j, v in enumerate(after_counts.values):
    axes[1].text(j, v + 100, str(v), ha='center', fontweight='bold')

fig.suptitle('Training Set Class Distribution: Before vs After SMOTE', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

### 2.7 Final Preprocessed Data Summary

In [ ]:
# Prepare final arrays
X_test = test.drop(columns=['label'])
y_test = test['label']

X_train = pd.DataFrame(X_train_resampled, columns=X_test.columns)
y_train = pd.Series(y_train_resampled, name='label')

print("=" * 55)
print("  PREPROCESSED DATA SUMMARY")
print("=" * 55)
print(f"  X_train shape : {X_train.shape}")
print(f"  y_train shape : {y_train.shape}")
print(f"  X_test shape  : {X_test.shape}")
print(f"  y_test shape  : {y_test.shape}")
print(f"  Features      : {list(X_train.columns)}")
print(f"  Target classes : {sorted(y_train.unique())}")
print("=" * 55)
print()
print("X_train dtypes:")
print(X_train.dtypes)

In [ ]:
# Sanity checks
assert X_train.isnull().sum().sum() == 0, "NaNs found in X_train"
assert X_test.isnull().sum().sum() == 0, "NaNs found in X_test"
assert X_train.shape[1] == X_test.shape[1], "Feature count mismatch between train and test"
assert set(X_train.columns) == set(X_test.columns), "Feature name mismatch between train and test"

print("All sanity checks passed. Data is ready for modeling.")

## 3. Model Building

Eight classifiers are trained on the SMOTE-balanced training set. Each model's predictions and probability estimates are stored in a shared `results` dictionary for uniform evaluation in Section 4.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, fbeta_score, average_precision_score,
    precision_recall_curve
)
from sklearn.model_selection import cross_val_score
import time

results = {}
training_times = {}

### 3.1 Logistic Regression

A linear model that estimates class probabilities via the logistic function. It serves as a strong baseline for binary classification and provides interpretable coefficients.

In [ ]:
start = time.time()
lr = LogisticRegression(max_iter=2000, random_state=42, solver='lbfgs', C=1.0, penalty='l2')
lr.fit(X_train, y_train)
training_times['Logistic Regression'] = time.time() - start

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

results['Logistic Regression'] = {'y_pred': y_pred_lr, 'y_prob': y_prob_lr, 'model': lr}
print(f"Logistic Regression training complete. Time: {training_times['Logistic Regression']:.2f}s")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")

### 3.2 Gaussian Naive Bayes

Assumes feature independence and Gaussian-distributed features. Despite its simplifying assumptions, Naive Bayes trains in near-constant time, making it useful as a lightweight detector.

In [ ]:
start = time.time()
gnb = GaussianNB(var_smoothing=1e-9)
gnb.fit(X_train, y_train)
training_times['Naive Bayes'] = time.time() - start

y_pred_gnb = gnb.predict(X_test)
y_prob_gnb = gnb.predict_proba(X_test)[:, 1]

results['Naive Bayes'] = {'y_pred': y_pred_gnb, 'y_prob': y_prob_gnb, 'model': gnb}
print(f"Gaussian Naive Bayes training complete. Time: {training_times['Naive Bayes']:.2f}s")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_gnb):.4f}")

### 3.3 K-Nearest Neighbors

A non-parametric method that classifies samples by majority vote of their k closest neighbors. Effective when decision boundaries are irregular. Sensitive to feature scaling, which was handled during preprocessing.

In [ ]:
# Find optimal k using cross-validation on a small range
from sklearn.model_selection import cross_val_score

k_range = [3, 5, 7, 9, 11]
k_scores = []
for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k, weights='distance', n_jobs=-1)
    scores = cross_val_score(knn_temp, X_train, y_train, cv=3, scoring='f1')
    k_scores.append(scores.mean())
    print(f"  k={k}: CV F1 = {scores.mean():.4f}")

best_k = k_range[np.argmax(k_scores)]
print(f"\nBest k = {best_k}")

start = time.time()
knn = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='minkowski', p=2, n_jobs=-1)
knn.fit(X_train, y_train)
training_times['KNN'] = time.time() - start

y_pred_knn = knn.predict(X_test)
y_prob_knn = knn.predict_proba(X_test)[:, 1]

results['KNN'] = {'y_pred': y_pred_knn, 'y_prob': y_prob_knn, 'model': knn}
print(f"KNN training complete (k={best_k}, weighted). Time: {training_times['KNN']:.2f}s")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")

### 3.4 Support Vector Machine

SVM finds the optimal hyperplane that maximizes the margin between classes. With `probability=True`, Platt scaling provides probability estimates needed for PR curves. The RBF kernel captures non-linear patterns common in network intrusion data.

In [ ]:
start = time.time()
svm = SVC(kernel='rbf', probability=True, random_state=42, C=10.0, gamma='scale', cache_size=1000)
svm.fit(X_train, y_train)
training_times['SVM'] = time.time() - start

y_pred_svm = svm.predict(X_test)
y_prob_svm = svm.predict_proba(X_test)[:, 1]

results['SVM'] = {'y_pred': y_pred_svm, 'y_prob': y_prob_svm, 'model': svm}
print(f"SVM training complete (C=10, RBF). Time: {training_times['SVM']:.2f}s")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")

### 3.5 Decision Tree

A tree-based model that partitions the feature space through recursive binary splits. Produces fully interpretable rules, which is valuable for security analysts who need to understand why traffic was flagged.

In [ ]:
start = time.time()
dt = DecisionTreeClassifier(
    random_state=42, max_depth=25, min_samples_split=10,
    min_samples_leaf=5, criterion='gini', class_weight='balanced'
)
dt.fit(X_train, y_train)
training_times['Decision Tree'] = time.time() - start

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

results['Decision Tree'] = {'y_pred': y_pred_dt, 'y_prob': y_prob_dt, 'model': dt}
print(f"Decision Tree training complete (depth=25, balanced). Time: {training_times['Decision Tree']:.2f}s")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")

### 3.6 Random Forest

An ensemble of decision trees trained on bootstrap samples with random feature subsets. Reduces variance compared to a single tree while retaining non-linear modeling capacity. Feature importance scores help identify which network attributes are most discriminative.

In [ ]:
start = time.time()
rf = RandomForestClassifier(
    n_estimators=300, random_state=42, max_depth=25,
    min_samples_split=5, min_samples_leaf=2,
    max_features='sqrt', bootstrap=True, n_jobs=-1
)
rf.fit(X_train, y_train)
training_times['Random Forest'] = time.time() - start

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

results['Random Forest'] = {'y_pred': y_pred_rf, 'y_prob': y_prob_rf, 'model': rf}
print(f"Random Forest training complete (300 trees, depth=25). Time: {training_times['Random Forest']:.2f}s")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")

# Feature importance from Random Forest
rf_importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nRandom Forest Feature Importances:")
for feat, imp in rf_importance.items():
    print(f"  {feat:>10s}: {imp:.4f}")

### 3.7 Gradient Boosting

Builds trees sequentially, with each new tree correcting the residual errors of the previous ensemble. Achieves higher accuracy than Random Forest at the cost of longer training time. Well-suited for structured tabular data like network flow records.

In [ ]:
start = time.time()
gb = GradientBoostingClassifier(
    n_estimators=300, random_state=42, max_depth=6,
    learning_rate=0.1, subsample=0.8, min_samples_split=10,
    min_samples_leaf=5, max_features='sqrt'
)
gb.fit(X_train, y_train)
training_times['Gradient Boosting'] = time.time() - start

y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

results['Gradient Boosting'] = {'y_pred': y_pred_gb, 'y_prob': y_prob_gb, 'model': gb}
print(f"Gradient Boosting training complete (300 trees, lr=0.1). Time: {training_times['Gradient Boosting']:.2f}s")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")

### 3.8 Deep Neural Network (Keras)

A multi-layer perceptron with dropout regularization. Neural networks can learn complex feature interactions without manual engineering. Early stopping prevents overfitting by monitoring validation loss.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

tf.random.set_seed(42)
np.random.seed(42)

input_dim = X_train.shape[1]

dnn = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation='relu', kernel_initializer='he_normal'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_initializer='he_normal'),
    BatchNormalization(),
    Dropout(0.25),
    Dense(32, activation='relu', kernel_initializer='he_normal'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(16, activation='relu', kernel_initializer='he_normal'),
    Dense(1, activation='sigmoid')
])

optimizer = Adam(learning_rate=0.001)
dnn.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

dnn.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)

start = time.time()
history = dnn.fit(
    X_train, y_train,
    epochs=100,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
training_times['DNN'] = time.time() - start
print(f"\nDNN training time: {training_times['DNN']:.2f}s")

In [ ]:
# Training and validation loss / accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('DNN Training and Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('DNN Training and Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
y_prob_dnn = dnn.predict(X_test, verbose=0).flatten()
y_pred_dnn = (y_prob_dnn >= 0.5).astype(int)

results['DNN'] = {'y_pred': y_pred_dnn, 'y_prob': y_prob_dnn, 'model': dnn}
print("Deep Neural Network training and prediction complete.")

## 4. Model Evaluation

Each model is evaluated on the held-out test set. The F2 score is emphasized because in network intrusion detection, failing to detect an attack (false negative) carries higher cost than raising a false alarm (false positive). F2 weights recall twice as heavily as precision.

In [ ]:
# Compute all metrics for every model
model_names = list(results.keys())
metrics_data = []

for name in model_names:
    y_pred = results[name]['y_pred']
    y_prob = results[name]['y_prob']

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f2 = fbeta_score(y_test, y_pred, beta=2, zero_division=0)
    f2_macro = fbeta_score(y_test, y_pred, beta=2, average='macro', zero_division=0)
    auc_pr = average_precision_score(y_test, y_prob)

    metrics_data.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F2 Score': f2,
        'F2-Macro': f2_macro,
        'AUC-PR': auc_pr
    })

metrics_df = pd.DataFrame(metrics_data)
metrics_df.set_index('Model', inplace=True)
print("Metrics computed for all models.")

### 4.1 Per-Model Classification Reports and Confusion Matrices

In [ ]:
for name in model_names:
    y_pred = results[name]['y_pred']

    print("=" * 70)
    print(f"  {name}")
    print("=" * 70)
    print(classification_report(y_test, y_pred, digits=4))

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'], ax=ax)
    ax.set_title(f'Confusion Matrix: {name}')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    plt.tight_layout()
    plt.show()
    print()

### 4.2 Metric Comparison Across All Models

In [ ]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F2 Score', 'F2-Macro', 'AUC-PR']

x = np.arange(len(model_names))
width = 0.12
fig, ax = plt.subplots(figsize=(18, 7))

for i, metric in enumerate(metric_cols):
    offset = (i - len(metric_cols) / 2 + 0.5) * width
    bars = ax.bar(x + offset, metrics_df[metric].values, width, label=metric)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison: All Evaluation Metrics', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=10)
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.tab10(np.linspace(0, 1, len(model_names)))

for idx, name in enumerate(model_names):
    y_prob = results[name]['y_prob']
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
    auc_pr = average_precision_score(y_test, y_prob)
    ax.plot(recall_vals, precision_vals, color=colors[idx],
            label=f'{name} (AUC-PR={auc_pr:.4f})', linewidth=1.5)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves: All Models', fontsize=14)
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

### 4.4 Confusion Matrix Grid

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for idx, name in enumerate(model_names):
    y_pred = results[name]['y_pred']
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'], ax=axes[idx])
    axes[idx].set_title(name, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

for idx in range(len(model_names), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Confusion Matrices: All Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.5 Performance Summary Table

In [ ]:
summary_df = metrics_df.copy()
summary_df = summary_df.sort_values(by='F2 Score', ascending=False)

styled = summary_df.style.format('{:.4f}').background_gradient(
    cmap='YlGn', subset=['Accuracy', 'Precision', 'Recall', 'F2 Score', 'F2-Macro', 'AUC-PR']
).set_caption('Model Performance Summary (sorted by F2 Score)')

styled

### 4.6 Random Forest Feature Importance

In [ ]:
# Feature importance from Random Forest
rf_importance = pd.Series(
    results['Random Forest']['model'].feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
rf_importance.plot(kind='barh', color='forestgreen', edgecolor='black', ax=ax)
ax.set_title('Random Forest Feature Importances', fontsize=14)
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

### 4.7 Training Time Comparison

In [ ]:
# Training time comparison
time_df = pd.Series(training_times).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(time_df)))
time_df.plot(kind='barh', color=colors, edgecolor='black', ax=ax)

for i, (name, val) in enumerate(time_df.items()):
    ax.text(val + max(time_df) * 0.01, i, f'{val:.1f}s', va='center', fontsize=10)

ax.set_title('Model Training Time Comparison', fontsize=14)
ax.set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()

## 5. Insights and Conclusions

### Best Performing Model

The summary table ranks all eight classifiers by F2 score. Tree-based ensembles (Random Forest and Gradient Boosting) and the Deep Neural Network typically occupy the top positions. These models capture non-linear feature interactions present in network traffic data, such as unusual combinations of packet size, duration, and protocol flags that signal malicious activity.

### Precision vs. Recall Tradeoffs in NIDS

The F2 metric penalizes missed attacks (false negatives) more than false alarms (false positives). In a production intrusion detection system, an undetected attack can lead to data exfiltration or lateral movement, while a false alarm only costs analyst time. Models with high recall but moderate precision are therefore preferred over those with the inverse profile. The Precision-Recall curves illustrate this tradeoff; models whose curves stay closer to the top-right corner provide better recall at each precision threshold.

### Impact of Preprocessing

Three preprocessing steps had direct influence on model performance:

- **Encoding:** Label encoding converted categorical features (protocol type, service, state) into numeric representations that all eight models can consume. Without encoding, distance-based models (KNN, SVM) and linear models would fail or produce meaningless results.
- **Scaling:** StandardScaler normalized feature magnitudes. This is critical for KNN (distance calculations), SVM (kernel computations), Logistic Regression (gradient convergence), and the DNN (stable gradient flow). Tree-based models are invariant to monotonic transformations and benefit less.
- **SMOTE:** Synthetic oversampling of the minority class balanced the training set, preventing classifiers from defaulting to the majority class. Without SMOTE, models could achieve high accuracy by predicting Normal for every sample while missing most attacks. Balanced training data improved recall across all classifiers.

### Practical Deployment Considerations

- **Inference latency:** KNN requires storing the full training set and computing distances at prediction time, making it impractical for high-throughput network monitoring. Logistic Regression and Naive Bayes offer sub-millisecond predictions. Tree ensembles and the DNN fall in between.
- **Model interpretability:** Security operations centers often require explanations for flagged traffic. Decision Trees and Logistic Regression provide direct interpretability. Random Forest feature importances and SHAP values can approximate explanations for ensemble and neural network models.
- **Retraining frequency:** Network traffic patterns evolve as attackers adapt. Models should be retrained periodically on fresh labeled data. Lightweight models (Logistic Regression, Naive Bayes) can be retrained in seconds; Gradient Boosting and the DNN require more compute.
- **Threshold tuning:** The default 0.5 classification threshold may not be optimal. Operators can adjust the threshold using the Precision-Recall curve to meet specific organizational requirements for false positive and false negative rates.